# Import Libraries

The following libraries are required for data preprocessing, visualization, model development, evaluation, and submission generation.

In [4]:
import os
import warnings
import time
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

# --------------------------------------------------------
# Scikit-learn Utilities
# --------------------------------------------------------

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

# --------------------------------------------------------
# SAS Viya Gradient Regressor
# --------------------------------------------------------

from sasviya.ml.tree import GradientBoostingRegressor

warnings.filterwarnings("ignore")

np.random.seed(42)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

sns.set_style("whitegrid")

print("Libraries imported successfully.")

Libraries imported successfully.


# Load  Dataset

In [5]:
train = pd.read_csv("VFL_2026_TRAIN_SET.csv")
test = pd.read_csv("VFL_2026_TEST_SET.csv")

print("=" * 60)
print("Competition Dataset")
print("=" * 60)

print(f"Training Set : {train.shape}")
print(f"Test Set     : {test.shape}")

display(train.head())

display(test.head())

Competition Dataset
Training Set : (100000, 46)
Test Set     : (15000, 45)


,ENCOUNTER_KEY,PATIENT_NUMBER,DOCTOR,ICU_DAYS,DEPARTMENT,DISCHARGED_TO,STANDARD_ORDERS_USED,NUM_CHRONIC_COND,DISCH_NURSE_ID,ORDER_SET_USED,ORDER_TOTAL_CHARGES,GENDER,ZIP,STATECODE,CITY,COUNTY_NAME,X,Y,REGION,RACE_CD,PATIENT_AGE,DIAGNOSIS_GROUP,ICD9_TARGET,MS_DRG_CODE,MS_DRG_DESC,DRG_APR_CODE,DRG_APR_DESC,DRG_APR_SEVERITY,DIAGNOSIS_SUBCAT_CODE,DIAGNOSIS_SUBCAT_DESC,DIAGNOSIS_ICD_CODE,DIAGNOSIS_LONG_DESC,PROCEDURE_SUBCAT_CODE,PROCEDURE_SUBCAT_DESC,PROCEDURE_ICD_CODE,PROCEDURE_LONG_DESC,DX_CODE,DX_GROUP,OPERATION_COUNT,MONITORING_HOURS,COMORBIDITY_INDEX,CARE_TEAM_SIZE,HOSPITAL,ADMIT_MTH,NUM_VISITS,ADMIT_LOS
0,300000000,574545941,256444,0,HEART,SKILLED NURSING FACIL,Y,0,2001,1,18768,F,33446,FL,Delray Beach,Palm Beach,-80.173388,26.451567,Region 11,Black,96,CHF,1,292,HEART FAILURE & SHOCK W CC,194,HEART FAILURE,3,428,HEART FAILURE,428.23,ACUTE ON CHRONIC SYSTOLIC HEART FAILURE,89,"INTERVIEW, EVALUATION, C",89.49,AUTOMATIC IMPLANTABLE CARDIOVERTERDEFIBRILLATO...,42823,Congestive heart failure; nonhypertensive [108.],1,32,4,4,Hosp 17,4,1,4
1,300000001,700724256,319034,1,HEART,SKILLED NURSING FACIL,N,1,8003,1,16584,F,33467,FL,Lake Worth,Palm Beach,-80.177700,26.606929,Region 1,Others,85,CHF,1,291,HEART FAILURE & SHOCK W MCC,194,HEART FAILURE,3,428,HEART FAILURE,428.23,ACUTE ON CHRONIC SYSTOLIC HEART FAILURE,38,"INCISION, EXCISION, AND",38.93,VENOUS CATHETERIZATION NOT ELSEWHERE CLASSIFIED,42823,Congestive heart failure; nonhypertensive [108.],0,0,0,5,Hosp 15,1,0,7
2,300000002,758601585,262051,0,HEART,HOME HEALTH AGENCY,Y,2,497217,1,30432,M,33455,FL,Hobe Sound,Martin,-80.143309,27.075482,Region 1,White,78,CHF,1,292,HEART FAILURE & SHOCK W CC,194,HEART FAILURE,2,428,HEART FAILURE,428.23,ACUTE ON CHRONIC SYSTOLIC HEART FAILURE,.,NaN,.,NaN,42823,Congestive heart failure; nonhypertensive [108.],1,47,3,3,Hosp 19,2,2,4
3,300000003,660243636,226751,1,PULMONARY,HOME HEALTH AGENCY,Y,1,4002,1,21517,M,33455,FL,Hobe Sound,Martin,-80.143309,27.075482,Region 8,White,87,CHF,1,292,HEART FAILURE & SHOCK W CC,194,HEART FAILURE,2,428,HEART FAILURE,428.00,CONGESTIVE HEART FAILURE UNSPECIFIED,.,NaN,.,NaN,4280,Congestive heart failure; nonhypertensive [108.],0,4,0,2,Hosp 14,2,2,5
4,300000004,572184863,284130,3,HEART,SKILLED NURSING FACIL,Y,1,326852,1,14583,M,33321,FL,Fort Lauderdale,Broward,-80.266294,26.205804,Region 11,White,81,CHF,0,291,HEART FAILURE & SHOCK W MCC,194,HEART FAILURE,2,404,HYPERTENSIVE HEART AND C,404.91,HYPERTENSIVE HEART AND CHRONIC KIDNEY DISEASE ...,.,NaN,.,NaN,40491,Hypertension with complications and secondary ...,0,11,2,3,Hosp 32,12,7,4


,ENCOUNTER_KEY,PATIENT_NUMBER,DOCTOR,ICU_DAYS,DEPARTMENT,DISCHARGED_TO,STANDARD_ORDERS_USED,NUM_CHRONIC_COND,DISCH_NURSE_ID,ORDER_SET_USED,ORDER_TOTAL_CHARGES,GENDER,ZIP,STATECODE,CITY,COUNTY_NAME,X,Y,REGION,RACE_CD,PATIENT_AGE,DIAGNOSIS_GROUP,ICD9_TARGET,MS_DRG_CODE,MS_DRG_DESC,DRG_APR_CODE,DRG_APR_DESC,DRG_APR_SEVERITY,DIAGNOSIS_SUBCAT_CODE,DIAGNOSIS_SUBCAT_DESC,DIAGNOSIS_ICD_CODE,DIAGNOSIS_LONG_DESC,PROCEDURE_SUBCAT_CODE,PROCEDURE_SUBCAT_DESC,PROCEDURE_ICD_CODE,PROCEDURE_LONG_DESC,DX_CODE,DX_GROUP,OPERATION_COUNT,MONITORING_HOURS,COMORBIDITY_INDEX,CARE_TEAM_SIZE,HOSPITAL,ADMIT_MTH,NUM_VISITS
0,300100000,553004402,297002,4,HEART,"ROUTINE DSCHG, HOME",Y,2,360014,1,33909,F,33445,FL,Delray Beach,Palm Beach,-80.102797,26.461329,Region 5,White,68,CHF,1,291,HEART FAILURE & SHOCK W MCC,194,HEART FAILURE,3,428,HEART FAILURE,428.00,CONGESTIVE HEART FAILURE UNSPECIFIED,99,OTHER NONOPERATIVE PROCE,99.22,INJECTION OF OTHER ANTIINFECTIVE,4280,Congestive heart failure; nonhypertensive [108.],2,68,6,4,Hosp 30,1,1
1,300100001,634863900,235415,0,HEART,"ROUTINE DSCHG, HOME",Y,1,100292,1,18657,M,35147,AL,Ragland,Saint Clair,-86.205444,33.758996,Region 3,Others,85,CHF,1,291,HEART FAILURE & SHOCK W MCC,194,HEART FAILURE,4,428,HEART FAILURE,428.43,ACUTE ON CHRONIC COMBINED SYSTOLIC AND DIASTOL...,88,OTHER DIAGNOSTIC RADIOLO,88.72,DIAGNOSTIC ULTRASOUND OF HEART,42843,Congestive heart failure; nonhypertensive [108.],3,67,8,6,Hosp 26,1,1
2,300100002,657483026,235415,10,TRANSPLANT,"ROUTINE DSCHG, HOME",Y,2,6003,0,38333,M,33446,FL,Delray Beach,Palm Beach,-80.173388,26.451567,Region 11,White,68,AMI,1,249,PERC CARDIOVASC PROC W NON-DRUG-ELUTING STENT ...,139,OTHER PNEUMONIA,2,486,PNEUMONIA ORGANISM UNSP,486.00,PNEUMONIA ORGANISM UNSPECIFIED,.,NaN,.,NaN,486,Pneumonia (except that caused by TB or STD) [1...,0,41,3,4,Hosp 10,1,1
3,300100003,922049972,235415,0,HEART,SKILLED NURSING FACIL,Y,0,320013,1,5996,M,34946,FL,Fort Pierce,Saint Lucie,-80.353492,27.482596,Region 10,White,74,CHF,1,293,HEART FAILURE & SHOCK W/O CC/MCC,194,HEART FAILURE,2,428,HEART FAILURE,428.00,CONGESTIVE HEART FAILURE UNSPECIFIED,88,OTHER DIAGNOSTIC RADIOLO,88.72,DIAGNOSTIC ULTRASOUND OF HEART,4280,Congestive heart failure; nonhypertensive [108.],0,17,0,3,Hosp 21,5,0
4,300100004,609359088,202927,0,HEART,"ROUTINE DSCHG, HOME",Y,4,18007,1,21206,F,33445,FL,Delray Beach,Palm Beach,-80.102797,26.461329,Region 7,White,69,CHF,1,292,HEART FAILURE & SHOCK W CC,194,HEART FAILURE,2,428,HEART FAILURE,428.00,CONGESTIVE HEART FAILURE UNSPECIFIED,.,NaN,.,NaN,4280,Congestive heart failure; nonhypertensive [108.],0,62,6,4,Hosp 36,11,9


# Dataset Overview



In [6]:
print("=" * 60)
print("Training Dataset Information")
print("=" * 60)

display(train.info())

print("\n")

print("=" * 60)
print("Training Dataset Summary Statistics")
print("=" * 60)

display(train.describe(include="all").T)

Training Dataset Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 46 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ENCOUNTER_KEY          100000 non-null  int64  
 1   PATIENT_NUMBER         100000 non-null  int64  
 2   DOCTOR                 100000 non-null  int64  
 3   ICU_DAYS               100000 non-null  int64  
 4   DEPARTMENT             100000 non-null  object 
 5   DISCHARGED_TO          100000 non-null  object 
 6   STANDARD_ORDERS_USED   100000 non-null  object 
 7   NUM_CHRONIC_COND       100000 non-null  int64  
 8   DISCH_NURSE_ID         100000 non-null  int64  
 9   ORDER_SET_USED         100000 non-null  int64  
 10  ORDER_TOTAL_CHARGES    100000 non-null  int64  
 11  GENDER                 100000 non-null  object 
 12  ZIP                    100000 non-null  int64  
 13  STATECODE              100000 non-null  object 
 14  CITY    

None



Training Dataset Summary Statistics


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ENCOUNTER_KEY,100000.0,NaN,NaN,NaN,300049999.5,28867.657797,300000000.0,300024999.75,300049999.5,300074999.25,300099999.0
PATIENT_NUMBER,100000.0,NaN,NaN,NaN,750402085.57081,144031876.51644,500000616.0,625802198.0,750262054.5,875106850.25,999994890.0
DOCTOR,100000.0,NaN,NaN,NaN,268049.11428,37456.350152,201620.0,235415.0,268058.0,297501.0,344581.0
ICU_DAYS,100000.0,NaN,NaN,NaN,2.7563,3.854618,0.0,0.0,1.0,4.0,29.0
DEPARTMENT,100000,10,HEART,54567,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DISCHARGED_TO,100000,11,"ROUTINE DSCHG, HOME",63099,NaN,NaN,NaN,NaN,NaN,NaN,NaN
STANDARD_ORDERS_USED,100000,2,Y,79982,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NUM_CHRONIC_COND,100000.0,NaN,NaN,NaN,1.02947,0.938856,0.0,0.0,1.0,1.0,4.0
DISCH_NURSE_ID,100000.0,NaN,NaN,NaN,166733.39955,158659.643526,1001.0,13005.0,145604.0,326852.0,827298.0
ORDER_SET_USED,100000.0,NaN,NaN,NaN,0.80383,0.397101,0.0,1.0,1.0,1.0,1.0


# Data Quality

In [7]:
# ==========================================================
# Missing Value Summary
# ==========================================================

train_missing = (
    train.isnull()
         .sum()
         .to_frame("Missing Values")
)

train_missing["Percentage (%)"] = (
    train_missing["Missing Values"] /
    len(train) * 100
).round(2)

train_missing = train_missing[
    train_missing["Missing Values"] > 0
].sort_values(
    "Missing Values",
    ascending=False
)

print("=" * 60)
print("Training Dataset Missing Values")
print("=" * 60)

display(train_missing)

print()

test_missing = (
    test.isnull()
        .sum()
        .to_frame("Missing Values")
)

test_missing["Percentage (%)"] = (
    test_missing["Missing Values"] /
    len(test) * 100
).round(2)

test_missing = test_missing[
    test_missing["Missing Values"] > 0
].sort_values(
    "Missing Values",
    ascending=False
)

print("=" * 60)
print("Test Dataset Missing Values")
print("=" * 60)

display(test_missing)

Training Dataset Missing Values


,Missing Values,Percentage (%)
PROCEDURE_SUBCAT_DESC,32072,32.07
PROCEDURE_LONG_DESC,32072,32.07



Test Dataset Missing Values


,Missing Values,Percentage (%)
PROCEDURE_SUBCAT_DESC,4778,31.85
PROCEDURE_LONG_DESC,4778,31.85


In [8]:
# ==========================================================
# Duplicate Records
# ==========================================================

print("=" * 60)
print("Duplicate Record Check")
print("=" * 60)

print(f"Training duplicate rows : {train.duplicated().sum()}")

print(f"Test duplicate rows     : {test.duplicated().sum()}")

if "ENCOUNTER_KEY" in train.columns:
    print(
        "Duplicate ENCOUNTER_KEY (Train):",
        train["ENCOUNTER_KEY"].duplicated().sum()
    )

if "ENCOUNTER_KEY" in test.columns:
    print(
        "Duplicate ENCOUNTER_KEY (Test):",
        test["ENCOUNTER_KEY"].duplicated().sum()
    )

Duplicate Record Check
Training duplicate rows : 0
Test duplicate rows     : 0
Duplicate ENCOUNTER_KEY (Train): 0
Duplicate ENCOUNTER_KEY (Test): 0


In [9]:
# ==========================================================
# Invalid Value Checks
# ==========================================================

def invalid_value_checks(df):

    checks = []

    temp = df.copy()

    numeric_cols = [
        "PATIENT_AGE",
        "ICU_DAYS",
        "NUM_VISITS",
        "NUM_CHRONIC_COND",
        "OPERATION_COUNT",
        "ADMIT_LOS"
    ]

    for col in numeric_cols:

        if col in temp.columns:

            temp[col] = pd.to_numeric(
                temp[col],
                errors="coerce"
            )

    def add_check(name, condition):

        invalid = int(condition.sum())

        pct = round(
            invalid / len(temp) * 100,
            2
        )

        checks.append({
            "Check": name,
            "Invalid Count": invalid,
            "Invalid (%)": pct,
            "Status": (
                "PASS ✅"
                if invalid == 0
                else "REVIEW ⚠️"
            )
        })

    if "PATIENT_AGE" in temp.columns:
        add_check(
            "PATIENT_AGE < 0",
            temp["PATIENT_AGE"] < 0
        )

    if "ICU_DAYS" in temp.columns:
        add_check(
            "ICU_DAYS < 0",
            temp["ICU_DAYS"] < 0
        )

    if "NUM_VISITS" in temp.columns:
        add_check(
            "NUM_VISITS < 0",
            temp["NUM_VISITS"] < 0
        )

    if "NUM_CHRONIC_COND" in temp.columns:
        add_check(
            "NUM_CHRONIC_COND < 0",
            temp["NUM_CHRONIC_COND"] < 0
        )

    if "OPERATION_COUNT" in temp.columns:
        add_check(
            "OPERATION_COUNT < 0",
            temp["OPERATION_COUNT"] < 0
        )

    if "ADMIT_LOS" in temp.columns:
        add_check(
            "ADMIT_LOS < 0",
            temp["ADMIT_LOS"] < 0
        )

    if (
        "ICU_DAYS" in temp.columns and
        "ADMIT_LOS" in temp.columns
    ):

        add_check(
            "ICU_DAYS > ADMIT_LOS",
            temp["ICU_DAYS"] >
            temp["ADMIT_LOS"]
        )

    return pd.DataFrame(checks)

display(
    invalid_value_checks(train)
)

,Check,Invalid Count,Invalid (%),Status
0,PATIENT_AGE < 0,0,0.00,PASS ✅
1,ICU_DAYS < 0,0,0.00,PASS ✅
2,NUM_VISITS < 0,0,0.00,PASS ✅
3,NUM_CHRONIC_COND < 0,0,0.00,PASS ✅
4,OPERATION_COUNT < 0,0,0.00,PASS ✅
5,ADMIT_LOS < 0,0,0.00,PASS ✅
6,ICU_DAYS > ADMIT_LOS,10665,10.66,REVIEW ⚠️


In [10]:
# ==========================================================
# Numerical Summary
# ==========================================================

numeric_summary = (
    train
    .select_dtypes(include=["number"])
    .describe()
    .T
)

display(numeric_summary)

,count,mean,std,min,25%,50%,75%,max
ENCOUNTER_KEY,100000.0,3.000500e+08,2.886766e+04,3.000000e+08,3.000250e+08,3.000500e+08,3.000750e+08,3.001000e+08
PATIENT_NUMBER,100000.0,7.504021e+08,1.440319e+08,5.000006e+08,6.258022e+08,7.502621e+08,8.751069e+08,9.999949e+08
DOCTOR,100000.0,2.680491e+05,3.745635e+04,2.016200e+05,2.354150e+05,2.680580e+05,2.975010e+05,3.445810e+05
ICU_DAYS,100000.0,2.756300e+00,3.854618e+00,0.000000e+00,0.000000e+00,1.000000e+00,4.000000e+00,2.900000e+01
NUM_CHRONIC_COND,100000.0,1.029470e+00,9.388559e-01,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,4.000000e+00
DISCH_NURSE_ID,100000.0,1.667334e+05,1.586596e+05,1.001000e+03,1.300500e+04,1.456040e+05,3.268520e+05,8.272980e+05
ORDER_SET_USED,100000.0,8.038300e-01,3.971006e-01,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
ORDER_TOTAL_CHARGES,100000.0,2.196222e+04,1.217863e+04,2.000000e+02,1.369600e+04,2.077750e+04,2.867400e+04,1.005600e+05
ZIP,100000.0,3.397352e+04,5.255866e+03,2.215000e+04,3.316100e+04,3.344400e+04,3.345500e+04,7.821900e+04
X,100000.0,-8.108273e+01,2.433410e+00,-9.838098e+01,-8.063284e+01,-8.021742e+01,-8.014331e+01,-7.705166e+01


# Feature Preparation



In [11]:
# ==========================================================
# Feature Configuration
# ==========================================================

TARGET = "ADMIT_LOS"

# ----------------------------------------------------------
# Identifier Columns
# ----------------------------------------------------------

IDENTIFIER_COLUMNS = [

    "ENCOUNTER_KEY",

    "PATIENT_NUMBER",

    "DOCTOR",

    "DISCH_NURSE_ID",

]

# ----------------------------------------------------------
# Leakage Columns
# ----------------------------------------------------------

LEAKAGE_COLUMNS = [

    "DISCHARGE_DATE",

]

# ----------------------------------------------------------
# Optional Feature Removal
#
# ONLY edit this list when experimenting.
# ==========================================================

OPTIONAL_REMOVE_COLUMNS = [

    # Geography
    # "ZIP",
    # "CITY",
    # "COUNTY_NAME",
    # "STATECODE",
    # "X",
    # "Y",

    # Descriptions
    # "DIAGNOSIS_LONG_DESC",
    # "PROCEDURE_LONG_DESC",

]

In [12]:
# ==========================================================
# Dataset Feature Summary
# ==========================================================

feature_summary = pd.DataFrame({

    "Feature": train.columns,

    "Data Type": train.dtypes.astype(str).values,

    "Missing Values": train.isnull().sum().values,

    "Missing (%)": (
        train.isnull().sum()
        / len(train)
        * 100
    ).round(2).values,

    "Unique Values": train.nunique(dropna=False).values

})

feature_summary["Type"] = np.where(

    feature_summary["Data Type"].str.contains(
        "int|float"
    ),

    "Numeric",

    "Categorical"

)

display(
    feature_summary.sort_values(
        "Unique Values",
        ascending=False
    )
)

,Feature,Data Type,Missing Values,Missing (%),Unique Values,Type
0,ENCOUNTER_KEY,int64,0,0.00,100000,Numeric
1,PATIENT_NUMBER,int64,0,0.00,100000,Numeric
10,ORDER_TOTAL_CHARGES,int64,0,0.00,39160,Numeric
17,Y,float64,0,0.00,261,Numeric
16,X,float64,0,0.00,261,Numeric
12,ZIP,int64,0,0.00,242,Numeric
39,MONITORING_HOURS,int64,0,0.00,213,Numeric
14,CITY,object,0,0.00,163,Categorical
2,DOCTOR,int64,0,0.00,98,Numeric
8,DISCH_NURSE_ID,int64,0,0.00,93,Numeric


In [13]:
# ==========================================================
# High Cardinality Features
# ==========================================================

high_cardinality = feature_summary[

    (feature_summary["Type"]=="Categorical")

    &

    (feature_summary["Unique Values"]>100)

].sort_values(
    "Unique Values",
    ascending=False
)

display(high_cardinality)

,Feature,Data Type,Missing Values,Missing (%),Unique Values,Type
14,CITY,object,0,0.0,163,Categorical


In [14]:
# ==========================================================
# Numeric Features
# ==========================================================

numeric_features_summary = feature_summary[
    feature_summary["Type"]=="Numeric"
]

display(numeric_features_summary)

,Feature,Data Type,Missing Values,Missing (%),Unique Values,Type
0,ENCOUNTER_KEY,int64,0,0.0,100000,Numeric
1,PATIENT_NUMBER,int64,0,0.0,100000,Numeric
2,DOCTOR,int64,0,0.0,98,Numeric
3,ICU_DAYS,int64,0,0.0,26,Numeric
7,NUM_CHRONIC_COND,int64,0,0.0,5,Numeric
8,DISCH_NURSE_ID,int64,0,0.0,93,Numeric
9,ORDER_SET_USED,int64,0,0.0,2,Numeric
10,ORDER_TOTAL_CHARGES,int64,0,0.0,39160,Numeric
12,ZIP,int64,0,0.0,242,Numeric
16,X,float64,0,0.0,261,Numeric


In [15]:
# ==========================================================
# Candidate Features for Investigation
# ==========================================================

candidate_features = feature_summary[

    (feature_summary["Unique Values"] > 100)

    |

    (feature_summary["Missing (%)"] > 20)

].sort_values(

    "Unique Values",

    ascending=False

)

display(candidate_features)

,Feature,Data Type,Missing Values,Missing (%),Unique Values,Type
0,ENCOUNTER_KEY,int64,0,0.00,100000,Numeric
1,PATIENT_NUMBER,int64,0,0.00,100000,Numeric
10,ORDER_TOTAL_CHARGES,int64,0,0.00,39160,Numeric
16,X,float64,0,0.00,261,Numeric
17,Y,float64,0,0.00,261,Numeric
12,ZIP,int64,0,0.00,242,Numeric
39,MONITORING_HOURS,int64,0,0.00,213,Numeric
14,CITY,object,0,0.00,163,Categorical
35,PROCEDURE_LONG_DESC,object,32072,32.07,65,Categorical
33,PROCEDURE_SUBCAT_DESC,object,32072,32.07,26,Categorical


In [16]:
# ==========================================================
# Feature Engineering
# ==========================================================

feature_train = train.copy()

feature_test = test.copy()

# ----------------------------------------------------------
# Remove Leakage Variables
# ----------------------------------------------------------

feature_train.drop(

    columns=LEAKAGE_COLUMNS,

    inplace=True,

    errors="ignore"

)

feature_test.drop(

    columns=LEAKAGE_COLUMNS,

    inplace=True,

    errors="ignore"

)

# ----------------------------------------------------------
# Remove Optional Features
# ----------------------------------------------------------

feature_train.drop(

    columns=OPTIONAL_REMOVE_COLUMNS,

    inplace=True,

    errors="ignore"

)

feature_test.drop(

    columns=OPTIONAL_REMOVE_COLUMNS,

    inplace=True,

    errors="ignore"

)

# ----------------------------------------------------------
# Date Features
# ----------------------------------------------------------

for df in [feature_train, feature_test]:

    if "ADMIT_DATE" in df.columns:

        df["ADMIT_DATE"] = pd.to_datetime(

            df["ADMIT_DATE"],

            errors="coerce"

        )

        df["ADMIT_MONTH"] = (

            df["ADMIT_DATE"].dt.month

        )

        df["ADMIT_DAYOFWEEK"] = (

            df["ADMIT_DATE"].dt.dayofweek

        )

        df.drop(

            columns="ADMIT_DATE",

            inplace=True

        )

In [17]:
# ==========================================================
# Build Feature Matrix
# ==========================================================

# Keep only identifier columns that actually exist
existing_identifier_columns = [

    c
    for c in IDENTIFIER_COLUMNS
    if c in feature_train.columns

]

print("=" * 60)
print("Feature Matrix")
print("=" * 60)

print("Identifier columns removed:")
print(existing_identifier_columns)

# ----------------------------------------------------------
# Build Training Features
# ----------------------------------------------------------

X = feature_train.drop(

    columns=[TARGET] + existing_identifier_columns,

    errors="ignore"

)

y = feature_train[TARGET].copy()

# ----------------------------------------------------------
# Build Test Features
# ----------------------------------------------------------

X_test_full = feature_test.drop(

    columns=existing_identifier_columns,

    errors="ignore"

)

# ----------------------------------------------------------
# Ensure identical feature sets
# ----------------------------------------------------------

common_columns = sorted(

    set(X.columns)

    &

    set(X_test_full.columns)

)

X = X[common_columns]

X_test_full = X_test_full[common_columns]

# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------

print()

print(f"Training Features : {X.shape}")

print(f"Test Features     : {X_test_full.shape}")

print(f"Total Features    : {len(common_columns)}")

Feature Matrix
Identifier columns removed:
['ENCOUNTER_KEY', 'PATIENT_NUMBER', 'DOCTOR', 'DISCH_NURSE_ID']

Training Features : (100000, 41)
Test Features     : (15000, 41)
Total Features    : 41


In [18]:
# ==========================================================
# Feature Verification
# ==========================================================

verification = {

    "Target Removed":
        TARGET not in X.columns,

    "Discharge Date Removed":
        "DISCHARGE_DATE" not in X.columns,

    "Encounter Key Removed":
        "ENCOUNTER_KEY" not in X.columns,

    "Patient Number Removed":
        "PATIENT_NUMBER" not in X.columns,

    "Doctor Removed":
        "DOCTOR" not in X.columns,

    "Discharge Nurse Removed":
        "DISCH_NURSE_ID" not in X.columns,

}

display(

    pd.DataFrame(

        verification.items(),

        columns=[

            "Check",

            "Status"

        ]

    )

)

,Check,Status
0,Target Removed,True
1,Discharge Date Removed,True
2,Encounter Key Removed,True
3,Patient Number Removed,True
4,Doctor Removed,True
5,Discharge Nurse Removed,True


In [19]:
# ==========================================================
# Feature Verification
# ==========================================================

verification = {

    "Target Removed":
        TARGET not in X.columns,

    "Discharge Date Removed":
        "DISCHARGE_DATE" not in X.columns,

    "Encounter Key Removed":
        "ENCOUNTER_KEY" not in X.columns,

    "Patient Number Removed":
        "PATIENT_NUMBER" not in X.columns,

    "Doctor Removed":
        "DOCTOR" not in X.columns,

    "Discharge Nurse Removed":
        "DISCH_NURSE_ID" not in X.columns,

}

display(

    pd.DataFrame(

        verification.items(),

        columns=[

            "Check",

            "Status"

        ]

    )

)

,Check,Status
0,Target Removed,True
1,Discharge Date Removed,True
2,Encounter Key Removed,True
3,Patient Number Removed,True
4,Doctor Removed,True
5,Discharge Nurse Removed,True


# Train / Validation Split


In [20]:
# ==========================================================
# Train / Validation Split
# ==========================================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("=" * 60)
print("Training / Validation Split")
print("=" * 60)

print(f"Training Samples   : {len(X_train):,}")
print(f"Validation Samples : {len(X_val):,}")

print()

print("Training Shape :", X_train.shape)
print("Validation Shape:", X_val.shape)

Training / Validation Split
Training Samples   : 80,000
Validation Samples : 20,000

Training Shape : (80000, 41)
Validation Shape: (20000, 41)


# Preprocessing Pipeline

In [21]:
# ==========================================================
# Feature Types
# ==========================================================

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("=" * 60)
print("Feature Summary")
print("=" * 60)

print(f"Numeric Features     : {len(numeric_features)}")
print(f"Categorical Features : {len(categorical_features)}")
print(f"Total Features       : {len(X_train.columns)}")

Feature Summary
Numeric Features     : 21
Categorical Features : 20
Total Features       : 41


In [22]:
# ==========================================================
# Preprocessing Pipeline
# ==========================================================

preprocessor = ColumnTransformer(

    transformers=[

        (

            "numeric",

            Pipeline([

                (

                    "imputer",

                    SimpleImputer(
                        strategy="median"
                    )

                )

            ]),

            numeric_features

        ),

        (

            "categorical",

            Pipeline([

                (

                    "imputer",

                    SimpleImputer(
                        strategy="most_frequent"
                    )

                ),

                (

                    "encoder",

                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False
                    )

                )

            ]),

            categorical_features

        )

    ],

    remainder="drop"

)

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


# Gradient Regressor Baseline


In [23]:
def evaluate_gradient(
    X_train,
    y_train,
    X_val,
    y_val,
):

    # ------------------------------------------
    # Detect feature types
    # ------------------------------------------

    numeric_features = X_train.select_dtypes(
        include=["number"]
    ).columns.tolist()

    categorical_features = X_train.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    # ------------------------------------------
    # Build preprocessor
    # ------------------------------------------

    preprocessor = ColumnTransformer(

        transformers=[

            (
                "numeric",

                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(strategy="median")
                    )
                ]),

                numeric_features

            ),

            (
                "categorical",

                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),

                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False
                        )
                    )

                ]),

                categorical_features

            )

        ],

        remainder="drop"

    )

    # ------------------------------------------
    # Forest
    # ------------------------------------------

    model = GradientBoostingRegressor(
    random_state=42
)

    pipeline = Pipeline([

        ("preprocess", preprocessor),

        ("model", model)

    ])

    start = time.time()

    pipeline.fit(
        X_train,
        y_train
    )

    elapsed = time.time() - start

    predictions = pipeline.predict(
        X_val
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_val,
            predictions
        )
    )

    mae = mean_absolute_error(
        y_val,
        predictions
    )

    r2 = r2_score(
        y_val,
        predictions
    )

    return {

        "Pipeline": pipeline,

        "RMSE": rmse,

        "MAE": mae,

        "R²": r2,

        "Training Time": elapsed

    }

In [24]:
# ==========================================================
# Baseline Gradient
# ==========================================================

baseline = evaluate_gradient(

    X_train,
    y_train,

    X_val,
    y_val

)

print("=" * 70)
print("BASELINE GB")
print("=" * 70)

print(f"RMSE : {baseline['RMSE']:.4f}")
print(f"MAE  : {baseline['MAE']:.4f}")
print(f"R²   : {baseline['R²']:.4f}")
print(f"Time : {baseline['Training Time']:.2f} seconds")

BASELINE GB
RMSE : 2.0770
MAE  : 1.3647
R²   : 0.7482
Time : 33.78 seconds


# Gradient Boosting Hyperparameter Search

In [25]:
gb_experiments = [

    {
        "Experiment": "Current Best",
        "Parameters": {
            "n_estimators": 350,
            "learning_rate": 0.05,
            "max_depth": 5,
            "random_state": 42
        }
    },

    {
        "Experiment": "Subsample 0.7",
        "Parameters": {
            "n_estimators": 350,
            "learning_rate": 0.05,
            "max_depth": 5,
            "subsample": 0.7,
            "random_state": 42
        }
    },

    {
        "Experiment": "Subsample 0.9",
        "Parameters": {
            "n_estimators": 350,
            "learning_rate": 0.05,
            "max_depth": 5,
            "subsample": 0.9,
            "random_state": 42
        }
    },

]

In [26]:
results = []

pipelines = {}

print("=" * 70)
print("GRADIENT BOOSTING HYPERPARAMETER SEARCH")
print("=" * 70)

for exp in gb_experiments:

    model = GradientBoostingRegressor(
        **exp["Parameters"]
    )

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])

    start = time.time()

    pipeline.fit(
        X_train,
        y_train
    )

    elapsed = time.time() - start

    predictions = pipeline.predict(
        X_val
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_val,
            predictions
        )
    )

    mae = mean_absolute_error(
        y_val,
        predictions
    )

    r2 = r2_score(
        y_val,
        predictions
    )

    results.append({

        "Experiment": exp["Experiment"],

        "RMSE": rmse,

        "MAE": mae,

        "R²": r2,

        "Training Time": elapsed

    })

    pipelines[
        exp["Experiment"]
    ] = pipeline

    print(
        f"{exp['Experiment']:<30}"
        f"RMSE={rmse:.4f}"
    )

results_df = (
    pd.DataFrame(results)
      .sort_values("RMSE")
      .reset_index(drop=True)
)

results_df.index += 1

display(results_df.round(4))

GRADIENT BOOSTING HYPERPARAMETER SEARCH


Current Best                  RMSE=2.0415
Subsample 0.7                 RMSE=2.0400
Subsample 0.9                 RMSE=2.0380


,Experiment,RMSE,MAE,R²,Training Time
1,Subsample 0.9,2.0380,1.3435,0.7576,84.2954
2,Subsample 0.7,2.0400,1.3437,0.7571,74.6130
3,Current Best,2.0415,1.3456,0.7568,63.4315


In [27]:
# ==========================================================
# Improvement over Baseline
# ==========================================================

baseline_rmse = results_df.loc[
    results_df["Experiment"] == "Current Best",
    "RMSE"
].iloc[0]

results_df["RMSE Improvement"] = (

    baseline_rmse

    -

    results_df["RMSE"]

).round(4)

display(results_df)

,Experiment,RMSE,MAE,R²,Training Time,RMSE Improvement
1,Subsample 0.9,2.037954,1.343493,0.757618,84.295381,0.0035
2,Subsample 0.7,2.040032,1.343731,0.757124,74.612982,0.0015
3,Current Best,2.041486,1.345604,0.756777,63.431473,0.0000


In [28]:
# ==========================================================
# Best Feature Set
# ==========================================================

best_feature_set = results_df.iloc[0]

print("=" * 70)
print("BEST FEATURE SET")
print("=" * 70)

display(best_feature_set.to_frame())

BEST FEATURE SET


,1
Experiment,Subsample 0.9
RMSE,2.037954
MAE,1.343493
R²,0.757618
Training Time,84.295381
RMSE Improvement,0.0035


# Final

In [29]:
# ==========================================================
# Final Model Training & Submission
# ==========================================================

# Select the best hyperparameter configuration
best_result = results_df.iloc[0]
best_experiment = best_result["Experiment"]
best_pipeline = pipelines[best_experiment]

print("=" * 70)
print("FINAL MODEL")
print("=" * 70)

print(f"Best Experiment : {best_experiment}")
print(f"Best RMSE       : {best_result['RMSE']:.4f}")

# Train final model on full training data
best_pipeline.fit(X, y)

print("Generating predictions...")

predictions = best_pipeline.predict(X_test_full)
predictions = np.clip(predictions, 0, None)

submission = pd.DataFrame({
    "ENCOUNTER_KEY": test["ENCOUNTER_KEY"],
    "ADMIT_LOS": predictions
})

# Validation
assert len(submission) == len(test)
assert list(submission.columns) == ["ENCOUNTER_KEY", "ADMIT_LOS"]
assert submission["ADMIT_LOS"].notna().all()
assert submission["ENCOUNTER_KEY"].is_unique

submission.to_csv("submission.csv", index=False)
results_df.to_csv("gradient_boosting_results.csv", index=False)

print("\nSubmission successfully generated!")
print(f"Rows    : {len(submission)}")
print(f"Columns : {list(submission.columns)}")

display(submission.head())

FINAL MODEL
Best Experiment : Subsample 0.9
Best RMSE       : 2.0380


Generating predictions...

Submission successfully generated!
Rows    : 15000
Columns : ['ENCOUNTER_KEY', 'ADMIT_LOS']


,ENCOUNTER_KEY,ADMIT_LOS
0,300100000,9.086036
1,300100001,10.778382
2,300100002,9.807746
3,300100003,3.111594
4,300100004,3.691855
